# Loading and exploring the WRF Analysis Toolkit

In [ ]:
import wrf_analysis_toolkit as wat

In [ ]:
print(wat.__version__)

To get all the functions defined in the module, call:

In [ ]:
print(wat.__all__)

And you can get the documentation for that function by running:

In [ ]:
help(wat.diagnostic)

As you can see, the best way is to use a pre-defined Sensible Variable.

To see which variables are available, run:

In [ ]:
wat.SensibleVariables.get_sv_names()

# Defining a region to plot

First, lets define where our data comes from, and where we will save outputs.

In [ ]:
!pwd

In [ ]:
wrfout_dir="/mnt/projects/wrf-storm-workshop/data/wrfout/arwen_ctrl/d01"
output_dir="/home/mbcxpfh2/WRF_workshop/outputs"

Then, lets print a terrain plot with the `region_ticks` parameter set to `True`.

This will allow us to choose the projected map coordinates (bottom and right axes), which need to be used for the `region` parameter.

In [ ]:
f = wat.terrain(
    variable_name="TerrainElevation",
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    region_ticks=True,
)

print(f"{output_dir}/{f}.pdf")

Lets zoom in to that section by setting and passing the `region` parameter and re-plot, to make sure it is what we want

In [ ]:
region="-2.4e6,-.6e6,-4e5,8e5"

f = wat.terrain(
    variable_name="TerrainElevation",
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    region_ticks=True,
    region=region,
    file_tag="_region",
)

print(f"{output_dir}/{f}.pdf")

# Generating animations of diagnostics

We're now ready to generate our first diagnostic.

For test plots, it is important to choose a small time frame, otherwise the plotting will take too long.

We do this by defining `time_form` and `time_to`. These have to be defined as strings in the format "YYYY-MM-DD_HH:MM:SS".

We can also define `time_step`, a string formatted as "HH:MM:SS", that sets the gap in time between files that are loaded. For example, setting `time_step` to "03:00:00" will load data files 3 hours apart.

In [ ]:
time_from="2021-11-26_18:00:00"
time_to="2021-11-26_21:00:00"
time_step="03:00:00"

And finally, we pass all the patameters we've been choosing to our diagnostic function, as well as a `variable_name`:

In [ ]:
f=wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    region=region,
    variable_name="SimRadarReflectivity1km",
    windbarb_gap=25,
)

print(f"{output_dir}/{f}.mp4")

You may also wish to make static plots, without merging these into an MP4 file. This can be done by setting the `make_mp4` parameter to `False`.

Note that if `make_mp4` is `False`, `save_pdf_frames` must be `True` and/or `clean_png_frames` must be `False`, otherwise the job will fail because no output would be saved.

In [ ]:
f=wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    region=region,
    variable_name="SimRadarReflectivity1km",
    windbarb_gap=25,
    make_mp4=False,
    clean_png_frames=False,
    save_pdf_frames=True,
)

print(f"{output_dir}/_{f}/")

# Defining your own Sensible Variable

Sometimes the variable you want to plot is not defined, or you may want to tweak properties of the plot a bit further.

To do this, you can define your own Sensible Variable, and pass it to the diagnostic function.

Below is an example of a 4d variable (time + 3D), where a slice is taken using the interpolation variable set with `interpvar` at the value given with `interpvalue`.

Note that we import `get_cmap`, to be able to define a color map for the plot.

In [ ]:
from matplotlib.pyplot import get_cmap

new_sens_var = wat.SensibleVariables.svariable(
    wrfname="wspd",
    dim = 4,
    ptitle="Wind Speed 925hPa [m/s]",
    outfile="WindSpeed925",
    nticks=12,
    nlevs=12,
    range_min=0,
    range_max=50,
    windbarbs=True,
    windbarb_gap=40,
    interpvar="pressure",
    interpvalue=925,
    colormap=get_cmap("YlGnBu"),
)

f=wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    region=region,
    variable_name=new_sens_var.outfile,
    sens_var=new_sens_var,
)

print(f"{output_dir}/{f}.mp4")

# Adding an overlapping diagnostic to an existing Sensible Variable

In some cases, you will want to overlap two variables. This is done, for example, in the predefined Frontogenesis variables.

The scripts are equipped to generate an empty contour map, which is overlayed on top of a normally plotted diagnostic.

To do so, you need to modify the `overlap_sv` property of a Sensible Variable. Make sure you also set an appropriate title and file name.

To prevent issues, it is best if you create a deep copy of an existing sensible variable, or that you define a new sensible variable to be used.

In the example below, we copy and update an existing variable.

In [ ]:
from copy import deepcopy

my_new_var = deepcopy(wat.SensibleVariables.RelativeHumidity925)

my_new_var.overlap_sv="PotentialTemp925"
my_new_var.overlap_gap=1
my_new_var.overlap_cmap=get_cmap("coolwarm")
my_new_var.ptitle="RelativeHumidity + PotentialTemp at 925 hPa"
my_new_var.outfile="RelHum_PotTemp_925"

f_orig=wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    region=region,
    variable_name="RelativeHumidity925",
)

print(f"{output_dir}/{f_orig}.mp4")

f_overlapped=wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    region=region,
    variable_name=my_new_var.outfile,
    sens_var=my_new_var,
)

print(f"{output_dir}/{f_overlapped}.mp4")

# Making a Vertical Cross Section

The `Diagnostic` method can also be used to create vertical transects by setting `vcross` to `True`.

These require two latitude-longitude points located within the WRF domain, defined by `start_latlon` and `end_latlon`. It makes a contour plot of the field between the two locations, with pressure on the y-axis. (`start_latlon` and `end_latlon` can also be used to draw a line between two points on a 2D map plot, which is useful to show where the cross-section is being taken between).

The y-axis can be controlled using `plim_bottom` and `plim_top` to define the pressure level in hPa at the bottom and top of the y-axis. These default to 1000 and 100 hPa respectively. The number of labels on the y-axis can be changed using `plevs`, this defaults to 11.
We do this by defining `time_form` and `time_to`

In [ ]:
# Define lat-lon coordinates for the start and end points of the cross section
# These must be within the domain you are extracting data from
# These must be defined as tuple pairs of floats
start_latlon = (55,-2.0)
end_latlon = (50.0,2.0)

# (OPTIONAL) set the top, bottom, and number of pressure levels for the y-axis of the cross-section
# plot. These default to 100hPa, 1000hPa and 11 respectively
plim_top=200.
plim_bottom=900.
plevs=8

wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    variable_name="WindSpeed",
    file_tag="VertCrossSection",
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    vcross=True,
    start_latlon=start_latlon,
    end_latlon=end_latlon,
    plim_top=plim_top,
    plim_bottom=plim_bottom,
    plevs=plevs,
    save_pdf_frames=True
)

It is also possible to add line contours on top of the shaded contour, by modifying the `overlap_sv` property of a Sensible Variable.

In [ ]:
from copy import deepcopy
new_sens_var = deepcopy(wat.SensibleVariables.WindSpeed)
new_sens_var.ptitle = "Wind Speed (shaded) and Theta (contours)"
new_sens_var.outfile = "wspd_theta_cross"
new_sens_var.overlap_sv = "PotentialTemp"
new_sens_var.overlap_cmap = get_cmap("copper")
new_sens_var.overlap_gap = 5

wat.diagnostic(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    sens_var=new_sens_var,
    variable_name=new_sens_var.ptitle,
    file_tag="VertCrossSection",
    time_from=time_from,
    time_to=time_to,
    time_step=time_step,
    vcross=True,
    start_latlon=start_latlon,
    end_latlon=end_latlon,
    plim_top=plim_top,
    plim_bottom=plim_bottom,
    plevs=plevs,
    save_pdf_frames=True
)